# Week 6 — Part 03: How the job-skill analysis works

**Estimated time:** 60–90 minutes

## What success looks like (end of Part 03)

- You can name which stage actually identifies skills (the LLM stage) and which
  stages only prepare evidence.
- You can produce cheap, deterministic **keyword frequency hints** from job
  descriptions.
- You can assemble the evidence, build a structured prompt, validate the model
  output, and map it into the stable report schema.

## Learning Objectives

- Gather skill evidence from free text without sending full descriptions
- Build a structured prompt that asks for the theme skill fields
- Validate the LLM output and map it into `report.json`'s schema

**Lab tutorial**: [03_skill_analysis.md](./03_skill_analysis.md) — detailed walkthrough.

> This notebook runs **offline**: the keyword-frequency step is real, and the LLM
> step is mocked. The capstone replaces the mock with a real provider call (see
> `capstone_template/src/llm_interpretation.py`).

### The big idea

The "analysis" — extracting skills, clustering roles, ranking what to learn —
happens in the **LLM stage**. Everything before it exists to hand the model the
right **evidence**, cheaply and reproducibly:

```text
load -> profile -> compress (skill evidence) -> LLM -> validate -> report
```

We walk those steps on the real 60-row sample at `data/sample_job_postings.csv`.

## Step 1 — Gather skill evidence (keyword frequency hints)

Most skill signal is buried in `job_description` free text. A cheap first pass:
scan descriptions for known skills and count how many postings mention each.

This is fast, free, and explainable — but it only finds **known terms** and
cannot cluster roles or rank a learning path. We use it as *evidence for the
model*, not as the final answer.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

CSV_PATH = Path("data/sample_job_postings.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

jobs = pd.read_csv(CSV_PATH)

SKILL_KEYWORDS = [
    "python", "sql", "excel", "tableau", "power bi", "aws", "azure",
    "spark", "airflow", "docker", "kubernetes", "pandas", "scikit-learn",
    "tensorflow", "pytorch", "machine learning", "nlp", "etl",
    "data visualization", "statistics", "git", "communication", "dashboard",
]


def skill_frequency_hints(df, text_col="job_description", keywords=SKILL_KEYWORDS, top_k=15):
    """Count how many postings mention each known skill (cheap, deterministic)."""
    texts = df[text_col].dropna().astype(str).str.lower()
    counts = {}
    for kw in keywords:
        pattern = r"\b" + re.escape(kw) + r"\b"   # whole-word match
        n = int(texts.str.contains(pattern, regex=True).sum())
        if n:
            counts[kw] = n
    return dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:top_k])


hints = skill_frequency_hints(jobs)
print(f"Scanned {len(jobs)} postings. Top skill mentions:")
for skill, n in hints.items():
    print(f"  {skill:20} {n:>3} postings")

## Step 2 — Compress the evidence under the token budget

You cannot send 60 full descriptions to the model. Combine the cheap evidence
into a small, bounded object, then estimate its token cost (~4 chars/token) and
keep it well under ~2,000 tokens.

In [ ]:
def estimate_tokens(obj, chars_per_token=4.0):
    return int(len(json.dumps(obj)) / chars_per_token)


def existing_skill_strings(df, n=10):
    if "job_skills" not in df.columns:
        return []
    return df["job_skills"].dropna().astype(str).head(n).tolist()


evidence = {
    "n_postings": int(len(jobs)),
    "top_job_titles": {
        str(k): int(v) for k, v in jobs["job_title"].value_counts().head(8).to_dict().items()
    },
    "skill_keyword_hints": hints,
    "existing_skill_strings": existing_skill_strings(jobs),   # often empty in this dataset
    "sample_descriptions": (
        jobs["job_description"].dropna().astype(str).str.slice(0, 280).head(3).tolist()
    ),
    "note": "Compressed evidence only. Full descriptions are never sent in bulk.",
}

print(f"Evidence is ~{estimate_tokens(evidence)} tokens (budget: < 2000)")
(OUTPUT_DIR / "skill_evidence.json").write_text(json.dumps(evidence, indent=2, sort_keys=True), encoding="utf-8")
print("wrote:", OUTPUT_DIR / "skill_evidence.json")

## Step 3 — Build the structured prompt

The prompt asks for **JSON only**, with the theme skill fields, and tells the
model to use *only the supplied evidence* and to flag uncertainty in
`risk_notes`. That constraint is what keeps a skill analyzer from drifting into
generic career advice.

In [ ]:
THEME_FIELDS = [
    "common_skills", "common_tools", "role_clusters",
    "learning_priorities", "beginner_learning_path", "portfolio_project_ideas",
]


def build_prompt(evidence: dict) -> str:
    return f"""You are a careful job-market analyst.

Use ONLY the evidence below. Do not invent skills that are not supported by it.

Return ONLY valid JSON with these keys:
- summary: one short paragraph
- common_skills: list of frequently required technical skills
- common_tools: list of tools, platforms, or frameworks
- role_clusters: list of role families you see in the titles
- learning_priorities: ordered list of what a beginner should learn first
- beginner_learning_path: ordered list describing a sensible learning sequence
- portfolio_project_ideas: list of small projects that practice these skills
- risk_notes: list of caveats (small sample, truncated text, missing fields)

Evidence:
{json.dumps(evidence, indent=2, sort_keys=True)}
"""


prompt = build_prompt(evidence)
(OUTPUT_DIR / "llm_prompt.txt").write_text(prompt, encoding="utf-8")
print(prompt[:600], "\n...\n")
print("wrote:", OUTPUT_DIR / "llm_prompt.txt")

## Step 4 — Call the LLM (mock here, real in the capstone)

In the capstone, `call_llm()` makes a real provider call with timeout/retry.
Here we use a **debug-only mock** so the notebook runs offline. Either way, the
next thing you do is the same: **validate** the raw text before trusting it.

In [ ]:
def call_llm_mock(prompt: str) -> str:
    """DEBUG-ONLY stub. The capstone replaces this with a real provider call."""
    return json.dumps({
        "summary": "Analyst and engineering roles dominate; SQL, Python and dashboarding recur.",
        "common_skills": ["SQL", "Python", "data visualization", "communication", "statistics"],
        "common_tools": ["Tableau", "Power BI", "Excel", "AWS", "Git"],
        "role_clusters": ["Data Analyst", "Data/ML Engineer", "BI Developer"],
        "learning_priorities": ["SQL", "Python", "dashboarding", "basic statistics"],
        "beginner_learning_path": [
            "Learn SQL queries", "Learn Python + pandas",
            "Build a dashboard in Tableau/Power BI", "Document and present findings",
        ],
        "portfolio_project_ideas": [
            "A SQL + dashboard report over a public dataset",
            "An ETL script that cleans and profiles a CSV",
        ],
        "risk_notes": ["Small sample (60 rows)", "Descriptions truncated", "job_skills mostly empty"],
    })


REQUIRED_LLM_KEYS = ["summary", *THEME_FIELDS, "risk_notes"]


def validate_llm_output(raw_response: str) -> dict:
    """Parse JSON, check required keys, coerce missing lists to []."""
    data = json.loads(raw_response)              # in real code: wrap in try/except + one repair attempt
    missing = [k for k in REQUIRED_LLM_KEYS if k not in data]
    if missing:
        raise ValueError(f"LLM output missing keys: {missing}")
    for k in REQUIRED_LLM_KEYS:
        if k != "summary" and not isinstance(data[k], list):
            data[k] = []
    return data


raw_response = call_llm_mock(prompt)
(OUTPUT_DIR / "llm_raw_response.txt").write_text(raw_response, encoding="utf-8")
llm_output = validate_llm_output(raw_response)
print("validated. common_skills:", llm_output["common_skills"])
print("role_clusters:", llm_output["role_clusters"])

## Step 5 — Map into the stable report schema

The skill fields live **inside** `llm_interpretation`; the top-level report keys
stay fixed so every run looks the same to a reviewer. This mirrors
`capstone_template/src/report_builder.py`.

In [ ]:
def build_json_report(jobs, evidence, llm_output) -> dict:
    return {
        "metadata": {"source": "week6 lesson 03 (mock LLM)"},
        "dataset_summary": {"rows": int(len(jobs)), "columns": int(jobs.shape[1])},
        "data_quality": {"job_skills_non_empty": int(jobs["job_skills"].notna().sum())},
        "compression_summary": {"evidence_tokens_est": estimate_tokens(evidence)},
        "llm_interpretation": {
            "summary": llm_output["summary"],
            "common_skills": llm_output["common_skills"],
            "common_tools": llm_output["common_tools"],
            "role_clusters": llm_output["role_clusters"],
            "learning_priorities": llm_output["learning_priorities"],
            "beginner_learning_path": llm_output["beginner_learning_path"],
            "portfolio_project_ideas": llm_output["portfolio_project_ideas"],
        },
        "recommendations": llm_output["learning_priorities"],
        "risk_notes": llm_output["risk_notes"],
        "errors_or_warnings": [],
    }


report = build_json_report(jobs, evidence, llm_output)
(OUTPUT_DIR / "report.json").write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")

print("top-level report keys:", list(report.keys()))
print("\nllm_interpretation preview:")
print(json.dumps(report["llm_interpretation"], indent=2)[:600], "...")
print("\nwrote:", OUTPUT_DIR / "report.json")

## Keyword hints vs LLM — and why we use both

| | Keyword frequency | LLM interpretation |
|---|---|---|
| Cost | Free | API cost + latency |
| Explainable | Yes | Partially |
| Finds synonyms / new skills | No | Yes |
| Clusters, ranks, sequences | No | Yes |
| Can hallucinate | No | Yes |

Keyword hints are cheap, grounded **evidence**; the LLM turns that evidence into
clustered, ranked, human-readable guidance — while being told to stay within the
evidence.

### Self-check

- Which stage actually identified the skills? Which only prepared evidence?
- If you removed the keyword hints, what could the model no longer ground on?
- Are the skill fields nested in `llm_interpretation` with top-level keys fixed?
- What does your project do when `job_skills` is empty?

### Next step

Swap `call_llm_mock` for a real provider call (see
`capstone_template/src/llm_interpretation.py`) and keep everything else the same.